In [1]:
from datetime import datetime
import pandas as pd


class Backtester:
    """
    Runs the Bitcoin trading system against historical
    market data.

    This version uses the existing TradingAgent in
    paper-trading mode.
    """

    def __init__(self, agent):
        self.agent = agent
        self.results = []

    # =========================================================
    # RUN BACKTEST
    # =========================================================

    def run(self, df):
        """
        Run the trading agent against historical BTC data.

        Required columns:
            price
            atr
            rsi
            macd
            macd_signal
            volume_ratio

        Optional:
            timestamp
        """

        if df.empty:
            raise ValueError(
                "Historical DataFrame is empty."
            )

        required_columns = [
            "price",
            "atr",
            "rsi",
            "macd",
            "macd_signal",
            "volume_ratio"
        ]

        missing = [
            column
            for column in required_columns
            if column not in df.columns
        ]

        if missing:
            raise ValueError(
                f"Missing required columns: {missing}"
            )

        self.results = []

        for index, row in df.iterrows():

            price = float(row["price"])

            result = self.agent.process_market_data(
                price=price,
                atr=row["atr"],
                rsi=row["rsi"],
                macd=row["macd"],
                macd_signal=row["macd_signal"],
                volume_ratio=row["volume_ratio"]
            )

            portfolio = self.agent.get_portfolio()

            self.results.append({
                "index": index,
                "timestamp": row.get(
                    "timestamp",
                    datetime.now().isoformat()
                ),
                "price": price,
                "hybrid": result.get(
                    "hybrid_recommendation"
                ),
                "trade": result.get("trade"),
                "portfolio_value": portfolio.get(
                    "portfolio_value"
                ),
                "cash_usd": portfolio.get(
                    "cash_usd"
                ),
                "btc_quantity": portfolio.get(
                    "btc_quantity"
                ),
                "total_pnl": portfolio.get(
                    "total_pnl"
                ),
                "return_pct": portfolio.get(
                    "return_pct"
                )
            })

        return pd.DataFrame(self.results)

    # =========================================================
    # SUMMARY
    # =========================================================

    def summary(self):

        if not self.results:
            raise ValueError(
                "Run the backtest before requesting a summary."
            )

        df = pd.DataFrame(self.results)

        initial_value = float(
            df["portfolio_value"].iloc[0]
        )

        final_value = float(
            df["portfolio_value"].iloc[-1]
        )

        total_return_pct = (
            (final_value - initial_value)
            / initial_value
            * 100
        )

        return {
            "initial_portfolio_value": initial_value,
            "final_portfolio_value": final_value,
            "total_return_pct": total_return_pct,
            "final_pnl": float(
                df["total_pnl"].iloc[-1]
            ),
            "observations": len(df)
        }

In [3]:

from config_loader import (
    get_google_sheet_config,
    convert_config_values
)

rows = get_google_sheet_config()

config = convert_config_values(rows)

config

{'budget_usd': 10000,
 'dca_enabled': True,
 'dca_drop_pct': 3,
 'dca_amount': 500,
 'dca_interval_hours': 24,
 'atr_enabled': True,
 'atr_period': 14,
 'atr_multiplier': 1.5,
 'max_position_pct': 30,
 'portfolio_stop_pct': 25,
 'trading_mode': 'hybrid',
 'monitoring_interval_minutes': 30,
 'paper_trading': True,
 'llm_enabled': True,
 'max_position_usd': 2000,
 'max_trade_usd': 2000,
 'global_stop_loss_pct': 50,
 'max_active_trades': 20,
 'trading_fee_pct': 0.0015,
 'dca_buy_amount_usd': 500,
 'llm_min_confidence': 0.6}

In [5]:
from trading_agent import TradingAgent

agent = TradingAgent(config)

backtester = Backtester(agent)

risk_manager.py created successfully!


In [13]:
df = pd.read_csv('../data/btc_indicators.csv')

print(df.head())
print(df.columns.tolist())

                   timestamp      open      high       low     close  \
0  2026-08-29 05:30:00+00:00  77661.44  77717.57  77599.51  77635.43   
1  2026-08-29 06:00:00+00:00  77635.42  77641.27  77348.65  77404.83   
2  2026-08-29 06:30:00+00:00  77404.83  77486.92  77403.28  77459.98   
3  2026-08-29 07:00:00+00:00  77459.99  77534.72  77433.28  77520.47   
4  2026-08-29 07:30:00+00:00  77520.47  77672.41  77495.04  77619.00   

       volume      sma_20      sma_50        ema_12        ema_26     rsi_14  \
0   23.632645  77637.9650  78454.7616  77666.517761  77859.635544  41.331750   
1  152.731524  77633.5410  78408.1130  77626.258105  77825.946244  35.477430   
2   19.011998  77636.2905  78363.4380  77600.676858  77798.837634  37.748413   
3   19.520536  77642.4700  78317.0858  77588.337342  77778.217809  40.233178   
4   65.443273  77655.1460  78271.2842  77593.054674  77766.423897  44.144052   

         macd  macd_signal  macd_histogram      atr_14  volume_sma_20  \
0 -193.117783

In [14]:
print(df.shape)
print(df.columns.tolist())

(95, 18)
['timestamp', 'open', 'high', 'low', 'close', 'volume', 'sma_20', 'sma_50', 'ema_12', 'ema_26', 'rsi_14', 'macd', 'macd_signal', 'macd_histogram', 'atr_14', 'volume_sma_20', 'volume_ratio', 'price_change_pct']


In [17]:
df = df.rename(
    columns={
        "close": "price", "rsi_14": "rsi", "atr_14": "atr"
    }
)

In [18]:
results = backtester.run(df)

DCA BUY executed: $500.00 at $77,635.43


In [19]:
print(results.head())
print(backtester.summary())

   index                  timestamp     price hybrid  \
0      0  2026-08-29 05:30:00+00:00  77635.43   AUTO   
1      1  2026-08-29 06:00:00+00:00  77404.83   AUTO   
2      2  2026-08-29 06:30:00+00:00  77459.98   AUTO   
3      3  2026-08-29 07:00:00+00:00  77520.47   AUTO   
4      4  2026-08-29 07:30:00+00:00  77619.00   AUTO   

                                               trade  portfolio_value  \
0  {'timestamp': '2026-08-31T17:47:42.967273', 'a...      9999.250000   
1                                               None      9997.764853   
2                                               None      9998.120039   
3                                               None      9998.509616   
4                                               None      9999.144185   

   cash_usd  btc_quantity  total_pnl  return_pct  
0   9499.25       0.00644  -0.750000   -0.007500  
1   9499.25       0.00644  -2.235147   -0.022351  
2   9499.25       0.00644  -1.879961   -0.018800  
3   9499.25       0.